In [ ]:
# 입력 => 이미지, 좌표 (x1, y1, x2, y2), 라벨
# 출력 => 좌표, 라벨
from tensorflow import keras
from tensorflow.keras import layers

inputs = keras.Input(shape=(224, 224, 3))

x = layers.Conv2D(filters=32, kernel_size=(3,3), activation="relu")(inputs)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(filters=16, kernel_size=(3,3), activation="relu")(inputs)
x = layers.GlobalAveragePooling2D()(x)

# 객체 박스(회귀)
bbox = layers.Dense(units=4, activation="linear", name="bbox")(x)
# 객체 종류(분류)
ccls = layers.Dense(units=5, activation="softmax", name="ccls")(x)
# 입력은 1개 출력은 2개
model = keras.Model(inputs, [bbox, ccls])
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 222, 222,  │        448 │ input_layer_3[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 16)        │          0 │ conv2d_6[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bbox (Dense)        │ (None, 4)         │         68 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ccls (Dense)        │ (None, 5)         │         85 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 601 (2.35 KB)

 Trainable params: 601 (2.35 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.compile(
    optimizer="adam",
    loss={
        "bbox": "mse",
        "ccls": "sparse_categorical_crossentropy"
    },
    metrics={
        "bbox": "mae",
        "ccls": "accuracy"
    }
)

In [13]:
import numpy as np

# 100장의 이미지 (랜덤하게 생성한 가상 데이터)
x = np.random.rand(100, 224, 224, 3).astype(np.float32)
print(x.shape)

# bbox
y_bbox = np.random.rand(100, 4).astype(np.float32)
print(y_bbox.shape)

# ccls
y_ccls = np.random.randint(0, 5, size=(100,))
print(y_ccls.shape)

(100, 224, 224, 3)
(100, 4)
(100,)


In [14]:
history = model.fit(x, {"bbox":y_bbox, "ccls":y_ccls}, epochs=10, verbose=2)

Epoch 1/10
4/4 - 2s - 537ms/step - bbox_loss: 0.2529 - bbox_mae: 0.4203 - ccls_accuracy: 0.1700 - ccls_loss: 1.6763 - loss: 1.9101
Epoch 2/10
4/4 - 0s - 76ms/step - bbox_loss: 0.2096 - bbox_mae: 0.3748 - ccls_accuracy: 0.1500 - ccls_loss: 1.6348 - loss: 1.8556
Epoch 3/10
4/4 - 0s - 83ms/step - bbox_loss: 0.1631 - bbox_mae: 0.3436 - ccls_accuracy: 0.1600 - ccls_loss: 1.6322 - loss: 1.8158
Epoch 4/10
4/4 - 0s - 71ms/step - bbox_loss: 0.1495 - bbox_mae: 0.3246 - ccls_accuracy: 0.1600 - ccls_loss: 1.6193 - loss: 1.7869
Epoch 5/10
4/4 - 0s - 68ms/step - bbox_loss: 0.1269 - bbox_mae: 0.3091 - ccls_accuracy: 0.1600 - ccls_loss: 1.6262 - loss: 1.7651
Epoch 6/10
4/4 - 0s - 67ms/step - bbox_loss: 0.1237 - bbox_mae: 0.2982 - ccls_accuracy: 0.1600 - ccls_loss: 1.6153 - loss: 1.7509
Epoch 7/10
4/4 - 0s - 76ms/step - bbox_loss: 0.1132 - bbox_mae: 0.2875 - ccls_accuracy: 0.1600 - ccls_loss: 1.6192 - loss: 1.7356
Epoch 8/10
4/4 - 0s - 78ms/step - bbox_loss: 0.0987 - bbox_mae: 0.2785 - ccls_accuracy: 0

In [15]:
sample = np.random.rand(1,224,224,3).astype(np.float32)
p_bbox, p_ccls = model.predict(sample)
p_bbox.shape, p_ccls.shape

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step


((1, 4), (1, 5))

In [16]:
# 좌표 (4개의 실수), 종류
p_bbox[0], np.argmax(p_ccls[0], axis=0)

(array([0.5853038 , 0.5103651 , 0.4315625 , 0.41431803], dtype=float32),
 np.int64(4))